# 02 - Data Preparation and Features

Create the modeling frames used for the short-term and medium-term forecasts.

In [6]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass


import pandas as pd

from group5_energy.pipeline import (
    HALF_TARGET,
    DAILY_TARGET,
    add_history_lags,
    latest_clients,
    load_daily_history,
    load_daily_weather,
    load_half_hourly_history,
    load_holidays,
    load_hourly_weather,
    load_temperatures,
    prepare_daily_frame,
    prepare_half_hourly_frame,
)


In [7]:
half_history = load_half_hourly_history()
daily_history = load_daily_history()
weather_hourly = load_hourly_weather()
temperatures = load_temperatures()
weather_daily = load_daily_weather()
holidays = load_holidays()

half_clients = latest_clients(half_history, "DateTime")
daily_clients = latest_clients(daily_history, "Date")

half_features = prepare_half_hourly_frame(half_history, weather_hourly, temperatures, holidays, half_clients)
daily_features = prepare_daily_frame(daily_history, weather_daily, holidays, daily_clients)
half_features = add_history_lags(half_features, HALF_TARGET, "half_hourly")
daily_features = add_history_lags(daily_features, DAILY_TARGET, "daily")

half_features.shape, daily_features.shape

((80796, 34), (1683, 33))

In [8]:
half_features[[
    "Acorn", "DateTime", "Conso_moy", "temperature", "temperature_half_hour",
    "is_holiday", "half_hour_slot", "lag_48", "lag_336", "rolling_48_mean"
]].head(10)

,Acorn,DateTime,Conso_moy,temperature,temperature_half_hour,is_holiday,half_hour_slot,lag_48,lag_336,rolling_48_mean
0,ACORN-E,2012-06-30 22:00:00,0.208288,13.61,13.890,0,44,NaN,NaN,NaN
1,ACORN-E,2012-06-30 22:30:00,0.194951,13.61,13.890,0,45,NaN,NaN,NaN
2,ACORN-E,2012-06-30 23:00:00,0.171547,13.48,13.890,0,46,NaN,NaN,NaN
3,ACORN-E,2012-06-30 23:30:00,0.151595,13.48,13.890,0,47,NaN,NaN,NaN
4,ACORN-E,2012-07-01 00:00:00,0.140109,13.44,13.890,0,0,NaN,NaN,NaN
5,ACORN-E,2012-07-01 00:30:00,0.136640,13.44,13.890,0,1,NaN,NaN,NaN
6,ACORN-E,2012-07-01 01:00:00,0.123180,13.25,13.890,0,2,NaN,NaN,NaN
7,ACORN-E,2012-07-01 01:30:00,0.110136,13.25,13.335,0,3,NaN,NaN,NaN
8,ACORN-E,2012-07-01 02:00:00,0.101755,12.28,12.780,0,4,NaN,NaN,0.154556
9,ACORN-E,2012-07-01 02:30:00,0.102435,12.28,12.500,0,5,NaN,NaN,0.148689


In [9]:
daily_features[[
    "Acorn", "Date", "Conso_kWh", "temperatureMean", "is_holiday",
    "weekday", "lag_1", "lag_7", "rolling_7_mean"
]].head(10)

,Acorn,Date,Conso_kWh,temperatureMean,is_holiday,weekday,lag_1,lag_7,rolling_7_mean
0,ACORN-E,2012-07-01,8.501714,15.025,0,6,NaN,NaN,NaN
1,ACORN-E,2012-07-02,8.727650,17.120,0,0,8.501714,NaN,NaN
2,ACORN-E,2012-07-03,8.489889,18.695,0,1,8.727650,NaN,8.614682
3,ACORN-E,2012-07-04,8.173539,18.765,0,2,8.489889,NaN,8.573084
4,ACORN-E,2012-07-05,8.159257,16.740,1,3,8.173539,NaN,8.473198
5,ACORN-E,2012-07-06,8.343194,16.035,0,4,8.159257,NaN,8.410410
6,ACORN-E,2012-07-07,8.361754,16.525,0,5,8.343194,NaN,8.399207
7,ACORN-E,2012-07-08,8.988568,16.190,0,6,8.361754,8.501714,8.393857
8,ACORN-E,2012-07-09,8.627590,15.370,0,0,8.988568,8.727650,8.463407
9,ACORN-E,2012-07-10,8.420828,15.410,0,1,8.627590,8.489889,8.449113


In [10]:
missing_half = half_features.isna().mean().sort_values(ascending=False).head(15)
missing_daily = daily_features.isna().mean().sort_values(ascending=False).head(15)
pd.DataFrame({"half_hourly_missing_rate": missing_half}).join(
    pd.DataFrame({"daily_missing_rate": missing_daily}), how="outer"
)

,half_hourly_missing_rate,daily_missing_rate
apparentTemperature,0.000149,NaN
apparentTemperatureMax,NaN,0.003565
apparentTemperatureMin,NaN,0.003565
cloudCover,NaN,0.005348
dewPoint,0.000149,NaN
humidity,0.000149,0.003565
icon,0.000149,NaN
lag_14,NaN,0.024955
lag_336,0.012476,NaN
lag_48,0.001782,NaN


The first lag rows have missing values by construction. The model pipeline imputes these values during training and prediction.